# Business Issue: 품절을 막는 발주 의사결정

**문제**: 다음 주 수요를 반영하면 세 SKU가 안전재고 아래로 내려갑니다. 재고·수요·구매정책을 함께 확인해 예산 안의 발주 초안이 필요합니다.

이 실습은 `5_harness`의 specialist reranker와 Business Agent 프롬프트를 재사용합니다. 수량·금액 계산은 검증 가능한 Python 규칙으로 처리하고, 결과는 실제 발주 전에 사람 승인을 요구합니다.

<div style="margin: 18px 0 10px; padding: 7px 11px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 4px; font-size: 0.85em; font-weight: 700; color: #2e7d32;">🖥️  UI 실행</div>

## UI: 발주 의사결정 Playground

버튼을 눌러 읽기 전용 업무 데이터에 대해 권고안을 생성합니다. 이 실습은 실제 ERP 발주를 만들지 않으며, 결과는 항상 사람 승인을 기다리는 상태에서 멈춥니다.

In [ ]:
from pathlib import Path
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '5_harness').exists())
sys.path[:0] = [
    str(project_root / 'shared'),
    str(project_root / '5_harness'),
    str(project_root / '6_Business_issue' / 'src'),
]
from business_issue import load_case, solve_business_issue

def run_replenishment_case():
    result = solve_business_issue(load_case())
    recommendation = result['recommendation']
    summary = (
        f"총 발주 금액: {recommendation['total_order_cost_krw']:,}원\n"
        f"예산 잔여: {recommendation['remaining_budget_krw']:,}원\n"
        f"상태: {result['action']}\n"
        f"실제 write 수행: {result['write_performed']}\n\n"
        + result['analyst_writer_answer']
    )
    return result, summary

with gr.Blocks() as business_issue_ui:
    gr.Markdown('## Inventory Replenishment Decision')
    run = gr.Button('발주 초안 만들기', variant='primary')
    routing = gr.JSON(label='Routing and evidence')
    summary = gr.Textbox(label='Approval-required recommendation', lines=14)
    run.click(run_replenishment_case, outputs=[routing, summary])
business_issue_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1')


<div style="margin: 18px 0 10px; padding: 7px 11px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 4px; font-size: 0.85em; font-weight: 700; color: #1a73e8;">✅  체크포인트</div>

## 체크포인트

- specialist가 business-agent로 선택된다.
- 발주 초안의 상태가 approval_required이며 실제 write는 수행되지 않는다.
- 아래 코드로 근거·수량·예산 계산 과정을 자세히 확인할 수 있다.

<div style="margin: 18px 0 10px; padding: 7px 11px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 4px; font-size: 0.85em; font-weight: 700; color: #5f6368;">🔎  선택 심화</div>

## 선택: 코드로 같은 작업 보기

아래 셀은 UI가 사용하는 fixture, routing, 정책 계산을 단계별로 확인하는 자료입니다.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / '5_harness').exists():
    ROOT = ROOT.parents[1]
sys.path[:0] = [
    str(ROOT / 'shared'),
    str(ROOT / '5_harness'),
    str(ROOT / '6_Business_issue' / 'src'),
]

from business_issue import load_case, solve_business_issue


## 1. 읽기 전용 업무 데이터 확인

실제 환경에서는 Business Agent가 ERP/SCM과 구매정책 도구를 읽기 전용으로 호출합니다. 여기서는 동일한 경계를 fixture로 재현합니다.

In [ ]:
case = load_case()
print(case['request'])
print('예산:', f"{case['budget_krw']:,} {case['currency']}")
for item in case['items']:
    print(f"- {item['sku']}: 현재 {item['on_hand']:,}, 예측수요 {item['forecast_demand']:,}, 안전재고 {item['safety_stock']:,}")


## 2. Harness 흐름 실행

1. Harness reranker가 요청을 `business-agent`로 보냅니다.
2. Business Agent 형식의 근거 패킷이 예상재고와 안전재고 gap을 정리합니다.
3. 결정론적 정책 엔진이 MOQ·할인·예산을 계산합니다.
4. Analyst Writer 형식의 최종 답을 만들되, write tool은 호출하지 않습니다.

In [ ]:
result = solve_business_issue(case)
print('선택 specialist:', result['route']['selected'])
print('reranker:', result['route']['engine'])
print('Business Agent prompt 로드:', result['business_agent_prompt_loaded'])
print('\n근거 패킷')
for row in result['evidence_packet']['observations']:
    print(f"- {row['sku']}: 예상 {row['projected_stock']:,}, 안전재고 gap {row['gap_to_safety']:,} ({row['risk']})")


## 3. 발주 초안과 Human-in-the-loop

초안에는 MOQ와 할인 규칙이 반영됩니다. 예산을 통과해도 공급 가능일 확인과 담당자 승인이 끝나기 전에는 PO를 생성하지 않습니다.

In [ ]:
recommendation = result['recommendation']
for line in recommendation['lines']:
    discount = f"할인 {line['discount_rate']:.0%}" if line['discount_rate'] else '할인 없음'
    print(f"- {line['sku']} {line['quantity']:,}개 | {discount} | {line['net_cost_krw']:,}원")
print('합계:', f"{recommendation['total_order_cost_krw']:,}원")
print('예산 잔여:', f"{recommendation['remaining_budget_krw']:,}원")
print('상태:', result['action'])
print('실제 write 수행:', result['write_performed'])
print('\nAnalyst Writer 답변')
print(result['analyst_writer_answer'])


### 데모 메시지

- AI/Harness에는 **업무 요청을 올바른 specialist로 보내고, 읽기 전용 근거를 패킷화하며, 사용자 언어의 권고안을 만드는 일**을 맡깁니다.
- 안전재고·MOQ·할인·예산처럼 결정돼 있는 규칙은 **코드로 계산**하므로 검토와 재실행이 쉽습니다.
- ERP 발주 같은 변경 작업은 별도 승인된 write tool에만 연결합니다. 이 노트북은 그 직전의 `approval_required`에서 멈춥니다.